In [1]:
import os
import string
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Prepare splits

In [2]:
TARGET_CANCER = "C34"
SPLITS_PATH_TO_SAVE = f"datasets/{TARGET_CANCER}_splits.csv"
FILES = [
    f'datasets/{TARGET_CANCER}_matched_patients.csv',
]
df = pd.DataFrame()
for file in FILES:
    df = pd.concat([df, pd.read_csv(file, sep=',')], ignore_index=True)
df.shape

(1212, 10)

In [3]:
df["icd10_category"].value_counts()

icd10_category
C34    606
Name: count, dtype: int64

In [4]:
splits = train_test_split(df, test_size = 0.2, random_state=42, shuffle=True, stratify=df["icd10_category"].fillna("None"))
splits[0]["group"] = "train"
splits[1]["group"] = "test"
splits = pd.concat(splits)

In [5]:
if not os.path.exists(os.path.dirname(SPLITS_PATH_TO_SAVE)):
    os.makedirs(os.path.dirname(SPLITS_PATH_TO_SAVE))
splits.to_csv(SPLITS_PATH_TO_SAVE, index=False)

# Merge train-test split with features

In [3]:
DIR = "datasets"
NOSOLOGY = "C34"
SPLITS_PATH = f"{DIR}/{NOSOLOGY}_splits.csv"
SUFFIXES = [
    # "deepseek-ai_DeepSeek-V3_features_max",
    # "Qwen_Qwen3-235B-A22B-Instruct-2507_features_max",
    # "yandex_YandexGPT-5-Lite-8B-instruct_features_max",
    # "openai_gpt-oss-20b_Low_features_mean",
    # "openai_gpt-oss-20b_Medium_features_mean",
    # "openai_gpt-oss-20b_High_features_mean",
    # "openai_gpt-oss-20b_Low_features_max",
    # "openai_gpt-oss-20b_Medium_features_max",
    # "openai_gpt-oss-20b_High_features_max",
    # "openai_gpt-oss-120b_Low_features_mean",
    # "openai_gpt-oss-120b_Medium_features_mean",
    # "openai_gpt-oss-120b_High_features_mean",
    # "openai_gpt-oss-120b_Low_features_max",
    # "openai_gpt-oss-120b_Medium_features_max",
    # "openai_gpt-oss-120b_High_features_max",
    # "baseline_features",
    "random_4096_features"
]

for SUFFIX in tqdm(SUFFIXES):
    FEATURES_PATH = f"{DIR}/{NOSOLOGY}_{SUFFIX}.csv"
    DATASET_PATH = f"{DIR}/{NOSOLOGY}_{SUFFIX}.csv"

    splits = pd.read_csv(SPLITS_PATH)
    features = pd.read_csv(FEATURES_PATH)

    assert "group" not in features.columns, "group column already exists in features"
    df = pd.merge(
        features,
        splits[["subject_id", "group"]],
        on="subject_id"
    )
    df.to_csv(DATASET_PATH, index=False)

100%|██████████| 1/1 [00:08<00:00,  8.03s/it]
